# Denavit-Hartenberg Parameters for the 6R Robotic Arm

## Introduction

The **Denavit-Hartenberg (DH) convention** is a standardized method for describing the kinematics of robotic manipulators. It uses a compact set of 4 parameters per joint to define the transformation from one joint frame to the next, enabling systematic computation of forward kinematics and workspace analysis.

In this notebook, we walk through:
1. The 4 DH parameters and their geometric meaning
2. The homogeneous transformation matrix
3. Forward kinematics computation for the 6-DOF arm
4. Example poses and workspace visualization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import sys

# Add project root to path for importing custom modules
sys.path.insert(0, '/sessions/confident-stoic-bardeen/mnt/PBL_project')

## The Four DH Parameters

Each joint is characterized by 4 parameters:

| Parameter | Symbol | Meaning |
|-----------|--------|----------|
| **Link length** | `a` | Distance along joint axis (Z) from one frame to the next, along the direction of X_i |
| **Link twist** | `α` (alpha) | Rotation angle about X axis that aligns one frame's Z with the next's |
| **Joint offset** | `d` | Distance along Z axis between the previous joint and the current joint |
| **Joint angle** | `θ` (theta) | Rotation about Z axis (typically the actuated variable for revolute joints) |

For a 6-axis revolute arm, `θ_i` varies with joint position, while `a_i`, `α_i`, and `d_i` are fixed design parameters.

In [ ]:
# DH Parameters Table for the 6-DOF Robotic Arm
# Columns: [a (mm), alpha (deg), d (mm), theta_offset (deg)]
# theta values are added dynamically during FK computation

DH_TABLE = np.array([
    [0,      0,    150,   0],      # Joint 1: Vertical offset from base
    [200,   -90,      0,   0],    # Joint 2: First arm segment, perpendicular
    [150,     0,      0,   0],    # Joint 3: Second arm segment
    [0,      90,    100,   0],    # Joint 4: Wrist flexion
    [0,     -90,      0,   0],    # Joint 5: Wrist rotation
    [0,       0,     80,   0],    # Joint 6: End-effector offset
])

# Joint limits (in degrees) - typical industrial robot constraints
JOINT_LIMITS = np.array([
    [-180, 180],   # Joint 1
    [-135, 135],   # Joint 2
    [-180, 65],    # Joint 3
    [-180, 180],   # Joint 4
    [-120, 120],   # Joint 5
    [-360, 360],   # Joint 6
])

print("DH Parameters (a, α, d):")
print(DH_TABLE[:, :3])
print("\nJoint Limits (degrees):")
print(JOINT_LIMITS)

## The DH Transformation Matrix

The homogeneous transformation matrix from frame `i-1` to frame `i` is computed as:

$$T_{i}^{i-1} = \begin{bmatrix}
\cos(\theta_i) & -\sin(\theta_i)\cos(\alpha_i) & \sin(\theta_i)\sin(\alpha_i) & a_i\cos(\theta_i) \\
\sin(\theta_i) & \cos(\theta_i)\cos(\alpha_i) & -\cos(\theta_i)\sin(\alpha_i) & a_i\sin(\theta_i) \\
0 & \sin(\alpha_i) & \cos(\alpha_i) & d_i \\
0 & 0 & 0 & 1
\end{bmatrix}$$

This matrix encodes both the rotation and translation from frame `i-1` to frame `i`.

In [ ]:
def dh_transform(a, alpha, d, theta):
    """
    Compute the DH transformation matrix from frame i-1 to frame i.
    
    Args:
        a: Link length (mm)
        alpha: Link twist (radians)
        d: Joint offset (mm)
        theta: Joint angle (radians)
    
    Returns:
        T: 4x4 homogeneous transformation matrix
    """
    c_theta = np.cos(theta)
    s_theta = np.sin(theta)
    c_alpha = np.cos(alpha)
    s_alpha = np.sin(alpha)
    
    T = np.array([
        [c_theta, -s_theta * c_alpha,  s_theta * s_alpha, a * c_theta],
        [s_theta,  c_theta * c_alpha, -c_theta * s_alpha, a * s_theta],
        [0,        s_alpha,            c_alpha,            d],
        [0,        0,                  0,                  1]
    ])
    return T

# Compute T0->1 at theta=0
a1, alpha1, d1 = DH_TABLE[0, 0], np.deg2rad(DH_TABLE[0, 1]), DH_TABLE[0, 2]
theta1 = 0  # Home position

T01 = dh_transform(a1, alpha1, d1, theta1)

print("T0->1 (Joint 1 at theta=0):")
print(np.round(T01, 3))
print(f"\nEnd-effector position from joint 1: {T01[:3, 3]} mm")

## Forward Kinematics: Chaining Transformations

The position and orientation of the end-effector (frame 6) relative to the base (frame 0) is:

$$T_6^0 = T_1^0 \cdot T_2^1 \cdot T_3^2 \cdot T_4^3 \cdot T_5^4 \cdot T_6^5$$

By multiplying all 6 transformation matrices together, we get the complete end-effector pose for any joint configuration.

In [ ]:
def forward_kinematics(joint_angles):
    """
    Compute end-effector pose via forward kinematics.
    
    Args:
        joint_angles: Array of 6 joint angles in degrees
    
    Returns:
        T06: 4x4 transformation matrix (base to end-effector)
        positions: List of all joint frame positions (for visualization)
    """
    # Convert to radians
    theta_rad = np.deg2rad(joint_angles)
    
    # Start with identity (base frame)
    T = np.eye(4)
    positions = [T[:3, 3].copy()]
    
    # Chain all transformations
    for i in range(6):
        a, alpha, d = DH_TABLE[i, 0], np.deg2rad(DH_TABLE[i, 1]), DH_TABLE[i, 2]
        Ti = dh_transform(a, alpha, d, theta_rad[i])
        T = T @ Ti
        positions.append(T[:3, 3].copy())
    
    return T, positions

# Compute FK for home position (all zeros)
home_angles = np.zeros(6)
T06_home, positions_home = forward_kinematics(home_angles)

print("Home Position (all joints at 0°):")
print(f"End-effector position: {T06_home[:3, 3]} mm")
print(f"\nFull T0->6 matrix:")
print(np.round(T06_home, 2))

In [ ]:
# Compute FK for a sample pose
sample_angles = np.array([30, -45, 60, 0, 30, 0])  # [°, °, °, °, °, °]
T06_sample, positions_sample = forward_kinematics(sample_angles)

print(f"Sample Pose: {sample_angles}°")
print(f"End-effector position: {T06_sample[:3, 3]} mm")

# Visualize the arm configuration
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Plot joint positions
positions_sample = np.array(positions_sample)
ax.plot(positions_sample[:, 0], positions_sample[:, 1], positions_sample[:, 2], 
        'b-o', linewidth=2, markersize=8, label='Joint frames')

# Label joints
for i, pos in enumerate(positions_sample):
    ax.text(pos[0], pos[1], pos[2], f'J{i}', fontsize=10)

# Plot base frame
ax.quiver(0, 0, 0, 50, 0, 0, color='r', arrow_length_ratio=0.2, linewidth=2, label='X')
ax.quiver(0, 0, 0, 0, 50, 0, color='g', arrow_length_ratio=0.2, linewidth=2, label='Y')
ax.quiver(0, 0, 0, 0, 0, 50, color='b', arrow_length_ratio=0.2, linewidth=2, label='Z')

ax.set_xlabel('X (mm)')
ax.set_ylabel('Y (mm)')
ax.set_zlabel('Z (mm)')
ax.set_title(f'6-DOF Arm Configuration: {sample_angles}°')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nArm reach: {np.linalg.norm(T06_sample[:3, 3])} mm")

## Summary and Next Steps

In this notebook, we:

1. **Learned the DH convention**: Four parameters (a, α, d, θ) describe each joint's kinematics
2. **Built the transformation matrix**: Used the standard DH formula to compute frame-to-frame transforms
3. **Implemented forward kinematics**: Chained all 6 transforms to find end-effector position
4. **Visualized configurations**: Plotted arm poses in 3D space

**Next steps:**
- Use forward kinematics to solve **inverse kinematics** (finding joint angles for a desired EE position)
- Analyze **workspace** by sampling random joint configurations
- Plan **collision-free trajectories** through the workspace
- Validate kinematics against CAD model and physical measurements